# Module 2 · Lesson 07: Resume Customizer

Apply prompt engineering to a real career task: **tailoring resume bullet points**
to match a specific job description. This exercise ties together zero-shot, few-shot,
and constraint-based prompting.

## What you will learn
1. **System prompt design** for domain-specific tasks
2. **Few-shot examples** to control output style
3. **Constraint prompting** for length and format control
4. Iterative prompt refinement workflow

In [11]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown, clear_output
 
load_dotenv(Path.cwd().parent / "module_02_prompt_enginnering/.env")
 
from openai import OpenAI
 
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
 
OPENAI_MODEL = "gpt-4o-mini" 
if client:
    print(f"OpenAI client ready - using model {OPENAI_MODEL}")

OpenAI client ready - using model gpt-4o-mini


In [3]:
def ask_open_ai(prompt: str, system_content: str= None, temperature: float= 0.7, max_resp_tokens: int= 800, ai_model:  str="gpt-4o-mini") -> str:
    """
    Send a prompt to the OpenAI Chat Completions API and return the assistant's reply.

    This helper function builds a chat message list, optionally including a system
    instruction, sends it to the specified model, and returns the generated text
    content from the first response choice.

    Args:
        prompt (str): The user prompt to send to the model.
        system_content (str, optional): Optional system-level instruction that
            defines the assistant's behavior, tone, or constraints. If provided,
            it is added as the first message in the conversation. Defaults to None.
        temperature (float, optional): Sampling temperature used to control
            randomness in the model's response. Lower values produce more
            deterministic outputs; higher values produce more creative or varied
            outputs. Defaults to 0.7.
        max_resp_tokens (int, optional): Maximum number of tokens allowed in the
            generated response. Defaults to 800.
        ai_model (str, optional): The model name to use for the request.
            Defaults to "gpt-4o-mini".

    Returns:
        str: The text content of the assistant's first response message.

    Raises:
        Exception: Propagates any exception raised by the OpenAI client request
            (for example, authentication errors, rate limits, invalid parameters,
            or network-related failures).

    Example of use:
         reply = ask("Explain recursion in simple terms.")
         print(reply)

        reply = ask(
             prompt="Write a haiku about Python.",
             system_content="You are a poetic assistant.",
             temperature=0.9,
             max_resp_tokens=100,
             ai_model="gpt-4o-mini"
        )        
        print(reply)
    """
    msgs = []
    if system_content:
        msgs.append({"role": "system", "content": system_content})
    msgs.append({"role":"user", "content": prompt})
    resp = client.chat.completions.create(
        model=ai_model,
        messages=msgs,
        temperature=temperature,
        max_tokens=max_resp_tokens
    )
    return resp.choices[0].message.content

display(Markdown(f"**The function has run**"))

**The function has run**

---
## 1. Define the Input

Two inputs: the **job description** (target) and your **work experience** (source).

In [4]:
job_description = """Senior Python Developer - AI Team

We're looking for an experienced Python developer to join out AI tean.
You will build production ML pipelines, optimize model serving infrastructure and collaborate with data scientists.

Requirements:
- 5+ years of Python experience.
- Experience with FastAPI or Flask.
- ML/AI deployment (Docker, kubernetes).
- Strong testing practices (pytest CI/CD).
- Experience with cloud platforms (AWS/GCP).
"""

work_experience = """- Worked an a web application using Django for 3 years.
- Helped deploy models to production servers.
- Wrote some unit tests for the backed.
- Used AWS for hosting and S3 for storage.
- Collaborated with the data team on data pipelines.
- Built REST APIs for internal tools."""

---
## 2. Zero-Shot Approach

First attempt: just ask the model to rewrite, no examples.

In [5]:
ZERO_SHOT_PROMPT = """You are a senior hiring manager and resume expert.
Rewrite the candidate's experience bullets to better match the target job description.

Rules:
- Keep the same facts, but reframe using job's language.
- Start each bullet with a strong action verb.
- include qualifiable results where possible.
- Maximum 1 line per bullet point.
- Do NOT fabricate experience.
"""

zero_shot_resp = ask_open_ai(
    f"""Job description:
{job_description}

Original Experience:
{work_experience}

Rewritten bullets:""", 
system_content= ZERO_SHOT_PROMPT,
temperature=0.5
)

display(Markdown(f"### Zero-shot result:\n\n{zero_shot_resp}"))

### Zero-shot result:

- Developed and maintained robust web applications using Django for over 3 years, enhancing user experience and performance.  
- Deployed machine learning models to production servers, ensuring seamless integration and functionality.  
- Implemented comprehensive unit tests for backend services, improving code reliability and maintainability.  
- Leveraged AWS for hosting applications and S3 for efficient data storage solutions.  
- Collaborated closely with the data team to optimize data pipelines, driving efficiency in data processing workflows.  
- Engineered RESTful APIs for internal tools, facilitating smooth communication between services and enhancing system interoperability.  

---
## 3. Few-Shot Approach

Now let's provide **examples** of good rewrites to guide the style:

In [6]:
FEW_SHOT_SYSTEM = """You are a senior hiringmanager and resume expert.
Rewrite experience bullets to match a target job description.

Here are examples of good rewrites:

BEFORE: "Helped with database stuff."
AFTER: "Optimized PostgresSQL query performance, reducing p95 latency by 40% across 3 production services.

BEFORE: "Made some APIs"
AFTER: "Architected and deployed RESTful APIs serving 50K+ daily requests using FastAPI with async patterns"
 
BEFORE: "Did testing"
AFTER: "Established comprehensive test suite with 92% coverage using pytest, integrated into CI/CD pipeline"
 
Rules:
- Match the STYLE of the examples above.
- Use the job description's specific keywords.
- Start each bullet with a STRONG action verb.
- Add plausible metrics (but mark estimated ones with ~).
- Do NOT fabricate experience - only reframe existing facts.
"""

few_shot_resp = ask_open_ai(
    f"""Job description:
{job_description}

Original Experience:
{work_experience}

Rewritten bullets:""", 
system_content= FEW_SHOT_SYSTEM,
temperature=0.5
)

display(Markdown(f"### Few-shot result:\n\n{few_shot_resp}"))

### Few-shot result:

- Developed scalable web applications using Django, enhancing user experience and performance over 3 years, while implementing best practices in code quality and maintainability.
  
- Deployed machine learning models to production servers, optimizing model serving infrastructure with Docker and Kubernetes, achieving ~30% faster inference times.

- Established robust unit testing framework with pytest, achieving 85% code coverage and integrating it into the CI/CD pipeline to ensure consistent code quality.

- Leveraged AWS for hosting applications and utilized S3 for efficient data storage and retrieval, supporting a ~25% increase in application uptime.

- Collaborated closely with data scientists to design and implement data pipelines, streamlining data processing workflows and improving data accessibility for machine learning initiatives.

- Engineered RESTful APIs for internal tools, facilitating seamless communication between services and supporting ~10K daily requests using FastAPI.

---
## 4. Compare Results

In [7]:
compare = f""" ### Comparison
#### Original
{work_experience}

#### Zero-Shot Rewrite
{zero_shot_resp}

#### Few-Shot Rewrite
{few_shot_resp}
"""
display(Markdown(compare))


 ### Comparison
#### Original
- Worked an a web application using Django for 3 years.
- Helped deploy models to production servers.
- Wrote some unit tests for the backed.
- Used AWS for hosting and S3 for storage.
- Collaborated with the data team on data pipelines.
- Built REST APIs for internal tools.

#### Zero-Shot Rewrite
- Developed and maintained robust web applications using Django for over 3 years, enhancing user experience and performance.  
- Deployed machine learning models to production servers, ensuring seamless integration and functionality.  
- Implemented comprehensive unit tests for backend services, improving code reliability and maintainability.  
- Leveraged AWS for hosting applications and S3 for efficient data storage solutions.  
- Collaborated closely with the data team to optimize data pipelines, driving efficiency in data processing workflows.  
- Engineered RESTful APIs for internal tools, facilitating smooth communication between services and enhancing system interoperability.  

#### Few-Shot Rewrite
- Developed scalable web applications using Django, enhancing user experience and performance over 3 years, while implementing best practices in code quality and maintainability.
  
- Deployed machine learning models to production servers, optimizing model serving infrastructure with Docker and Kubernetes, achieving ~30% faster inference times.

- Established robust unit testing framework with pytest, achieving 85% code coverage and integrating it into the CI/CD pipeline to ensure consistent code quality.

- Leveraged AWS for hosting applications and utilized S3 for efficient data storage and retrieval, supporting a ~25% increase in application uptime.

- Collaborated closely with data scientists to design and implement data pipelines, streamlining data processing workflows and improving data accessibility for machine learning initiatives.

- Engineered RESTful APIs for internal tools, facilitating seamless communication between services and supporting ~10K daily requests using FastAPI.


In [8]:
from anthropic import Anthropic

claude_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
CLAUDE_MODEL = "claude-sonnet-4-6" 

if claude_client:
    print(f"Antropic client ready - using model {CLAUDE_MODEL}")

Antropic client ready - using model claude-sonnet-4-6


In [9]:
def ask_anthropic(prompt: str, system: str | None = None, temperature: float= 0.7, max_resp_tokens: int= 800, ai_model: str= "claude-sonnet-4-6", output_format: str | None = None) -> None:
    """
    Send a prompt to Anthropic using the streaming Messages API and return
    the complete generated text.

    Args:
        prompt (str):
            The user prompt to send to the model.
        system (str | None, optional):
            Optional system instruction for the model. In the streaming API
            for this SDK version, it is converted into the required list format.
            Defaults to None.
        temperature (float, optional):
            Controls randomness of the response. Defaults to 0.7.
        max_resp_tokens (int, optional):
            Maximum number of tokens in the generated response. Defaults to 800.
        ai_model (str, optional):
            The model name to use. Defaults to "claude-sonnet-4-6".
        output_format (str | None, optional):
            Optional output format for the streaming API, if supported by the
            installed SDK version. Defaults to None.

    Returns: None
    """

    params = {
        "model": ai_model,
        "max_tokens" : max_resp_tokens,
        "temperature": temperature,
        "messages": [
            {'role': 'user', 'content': prompt}
        ],
    }

    if system:
        params["system"] = [{"type": "text", "text": system}]

    if output_format:
        params["output_format"] = output_format
    
    with claude_client.messages.stream(**params) as stream:
        full_text = ""
        for text in stream.text_stream:
            full_text += text
            clear_output(wait=True)
            # display(Markdown(full_text)) # Only for this module need to be in comments.
    
    return full_text

display(Markdown(f"**The function has run**"))

**The function has run**

In [ ]:
coverage = ask_anthropic(
    f"""Analyze how well this resume matched the job description.

Job Description:
{job_description}

Resume Bullets:
{few_shot_resp}

Return JSON with:
- "matched_keywords": list of JD keywords found in resume.
- "missing_keywords": list of JD keywords NOT addressed.
- "match_score": percentage (0-100).
- "suggestions": list of 2-3 improvements.
""",
temperature=0
)

display(Markdown(coverage))

```json
{
  "matched_keywords": [
    "Python (Django - indirect match)",
    "FastAPI",
    "ML/AI deployment",
    "Docker",
    "Kubernetes",
    "pytest",
    "CI/CD",
    "AWS",
    "Collaborated with data scientists",
    "ML pipelines / data pipelines",
    "Model serving infrastructure"
  ],
  "missing_keywords": [
    "5+ years of Python experience (only 3 years explicitly stated)",
    "Flask",
    "GCP (Google Cloud Platform)",
    "Production ML pipelines (data pipelines mentioned, but not explicitly ML pipelines)"
  ],
  "match_score": 78,
  "suggestions": [
    "Explicitly state total years of Python experience upfront (e.g., '5+ years of Python development') since the JD requires 5+ years but the resume only mentions 3 years tied to Django, which is likely the biggest red flag for recruiters and ATS filters.",
    "Add a mention of GCP or multi-cloud experience if applicable, as the JD lists AWS/GCP as a requirement. Even familiarity or certification would help close this gap and broaden cloud coverage on the resume.",
    "Reframe the 'data pipelines' bullet to explicitly highlight production ML pipeline experience (e.g., 'Designed and deployed production ML pipelines...'), since that is a core responsibility in the JD and stronger alignment in wording will improve both ATS scoring and recruiter perception."
  ]
}
```

**Quick Summary:**
The resume is a **strong overall match** — hitting most technical requirements like FastAPI, Docker, Kubernetes, pytest, CI/CD, AWS, and data scientist collaboration. The two main risks are the **experience gap (3 vs. 5+ years)** and the **missing GCP mention**, which could cause automatic filtering before a human even reviews it.

In [14]:
coverage_zero = ask_anthropic(
    f"""Analyze how well this resume matched the job description.

Job Description:
{job_description}

Resume Bullets:
{zero_shot_resp}

Return ONLY JSON with:
- "matched_keywords": list of JD keywords found in resume.
- "missing_keywords": list of JD keywords NOT addressed.
- "match_score": percentage (0-100).
- "suggestions": list of 2-3 improvements.
""",
temperature=0
)

display(Markdown(coverage_zero))

```json
{
  "matched_keywords": [
    "Python (implied via Django)",
    "ML/AI deployment (production servers)",
    "Testing practices (unit tests)",
    "AWS",
    "Data pipelines",
    "RESTful APIs"
  ],
  "missing_keywords": [
    "FastAPI or Flask (uses Django instead)",
    "Docker",
    "Kubernetes",
    "CI/CD",
    "GCP",
    "pytest (specific framework not mentioned)",
    "5+ years Python experience (years not quantified)",
    "Model serving infrastructure"
  ],
  "match_score": 42,
  "suggestions": [
    "Explicitly mention Docker and Kubernetes experience if applicable, as container orchestration is a hard requirement for ML deployment in this role.",
    "Replace or supplement the generic 'unit tests' reference with pytest specifically, and add any CI/CD tools used (e.g., GitHub Actions, Jenkins) to directly address the testing and automation requirements.",
    "Quantify total Python experience (e.g., '6 years of Python development') and clarify ML deployment details such as model serving frameworks (e.g., TorchServe, TensorFlow Serving) to better align with the senior-level and AI-focused expectations."
  ]
}
```

---
## 5. Structured Output: Keyword Matching

Let's also check which job requirements are covered:

---
## Key Takeaways

| Technique | Effect |
|-----------|--------|
| **Zero-shot** | Quick but generic — good starting point |
| **Few-shot examples** | Controls output style and quality significantly |
| **Negative constraints** | "Do NOT fabricate" prevents hallucinated experience |
| **Keyword matching** | Structured JSON output to verify coverage |

> **Exercise:** Replace the sample job description with a real one you're interested in.
> Add your own work experience. Experiment with different few-shot examples to find
> the voice that works best for your industry.

---
**Next:** `08_delimeter_prompt` 